# T05 - Motor Trend Car Road Tests

**Nombre:** Valeria Estefanía Milke Loera

**Fecha:** 19-02-2026

**Expediente:** 7392228

In [2]:
import pandas as pd 
import numpy as np 
from sklearn.linear_model import LinearRegression
from scipy import stats
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

In [10]:
df0 = pd.read_excel(r"C:\Users\valer\Motor Trend Car Road Tests.xlsx")
df0.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


**1.1 Regresión con "mpg" como salida**

**Train-test-split 40% (muestra de 13 datos sin repetir)**

In [22]:
np.random.seed(42)
indices_train = np.random.choice(df0.index, size=13, replace=False)

train = df0.loc[indices_train].drop(columns=["model"])
test = df0.drop(indices_train).drop(columns=["model"])

In [23]:
y_train = X_train["mpg"]
y_test = X_test["mpg"]

X_train = train.drop(columns=["mpg"])
X_test = test.drop(columns=["mpg"])

In [24]:
cols_num = ["cyl","disp","hp","drat","wt","qsec","gear","carb"]
cols_bin = ["vs","am"]

**Ingeniería de características**

In [25]:
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[cols_num])
X_test_num = scaler.transform(X_test[cols_num])

In [26]:
X_train_scaled = np.concat([X_train_num, X_train[cols_bin].values], axis=1)
X_test_scaled = np.concat([X_test_num, X_test[cols_bin].values], axis=1)

**Regresión polinomial**

In [27]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

LinearRegression()

In [28]:
lr.intercept_

np.float64(17.322444956342583)

In [29]:
lr.coef_

array([ 14.50793042,  -7.60789639,  -1.15262483,   2.9988063 ,
         7.87922794,   2.7109519 ,  12.45288737, -15.834679  ,
        -1.7027746 ,   7.82441771])

**Interpretación de los coeficientes**

- El intercepto (17.32) es el valor base estimado de mpg cuando todas las variables son cero. 
- El coeficiente de cyl (14.51) indica que más cilindros aumentan el mpg, mientras que disp (-7.61) y hp (-1.15) muestran que motores más grandes y potentes reducen el rendimiento.
- drat (2.99), qsec (2.71) y gear (12.45) tienen efecto positivo sobre el mpg, es decir, se asocian con mayor eficiencia.
- carb (-15.83) reduce fuertemente el rendimiento, y vs (-1.70) tiene un efecto negativo leve.
- Finalmente, am (7.82) indica que la transmisión manual mejora el mpg respecto a la automática.

Algunos signos pueden no ser intuitivos debido al tamaño pequeño de muestra y posible correlación.

**Cálculo de estadístico R^2**

In [30]:
r2_train = lr.score(X_train_scaled, y_train)
r2_test = lr.score(X_test_scaled, y_test)

print("R2 train:", r2_train)
print("R2 test:", r2_test)

R2 train: 0.9770608924219335
R2 test: -1.479290396195823


**Regularización L2 (Ridge)**

In [31]:
lambdas = [0.01, 0.1, 1, 2, 10]
for l in lambdas:
    ridge = Ridge(alpha=l)
    ridge.fit(X_train_scaled, y_train)
    r2_train = ridge.score(X_train_scaled, y_train)
    r2_test = ridge.score(X_test_scaled, y_test)
    print("Lambda:", l)
    print("R2 train:", r2_train)
    print("R2 test:", r2_test)
    print("______________________________")

Lambda: 0.01
R2 train: 0.9723896638996921
R2 test: -0.19681647328223195
______________________________
Lambda: 0.1
R2 train: 0.9496144741282927
R2 test: 0.5313812970540165
______________________________
Lambda: 1
R2 train: 0.9176459206555573
R2 test: 0.720384671275101
______________________________
Lambda: 2
R2 train: 0.9011361907942816
R2 test: 0.7562924548483654
______________________________
Lambda: 10
R2 train: 0.8361691831679384
R2 test: 0.7731202270816397
______________________________


**Comparación de los R2**

Cuando λ es muy pequeño (0.01), el modelo se comporta casi como una regresión sin regularización. El R^2 de entrenamiento es muy alto (0.97), pero el R^2 de prueba es negativo, lo que indica un sobreajuste.
Al aumentar λ a 0.1 y 1, el R^2 de entrenamiento disminuye ligeramente, pero el R^2 de prueba mejora considerablemente, lo que muestra que la regularización está ayudando a reducir el sobreajuste, lo cual se puede comprobar con λ = 2 y λ = 10, pues el R^2 de prueba alcanza sus valores más altos (alrededor de 0.75–0.77), mientras que el R^2 de entrenamiento baja a valores más moderados. Esto indica que el modelo generaliza mejor cuando se penalizan más los coeficientes.
En conclusión, la regularización L2 mejora significativamente la capacidad de generalización del modelo, reduciendo el sobreajuste observado en la regresión sin penalización.

__________________________________

**1.2 Regresión con "qsec" como salida**

**Train-test-split**

In [40]:
np.random.seed(42)
indices_train = np.random.choice(df0.index, size=13, replace=False)

train = df0.loc[indices_train].drop(columns=["model"])
test = df0.drop(indices_train).drop(columns=["model"])

In [48]:
XX_train = train.drop(columns=["qsec"])
y_train = train["qsec"]

X_test = test.drop(columns=["qsec"])
y_test = test["qsec"]

In [49]:
cols_num = ["cyl","disp","hp","drat","wt","gear","carb"]
cols_bin = ["vs","am"]

**Ingeniería de características**

In [50]:
scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train[cols_num])
X_test_num = scaler.transform(X_test[cols_num])

X_train_scaled = np.concatenate([X_train_num, X_train[cols_bin].values], axis=1)
X_test_scaled = np.concatenate([X_test_num, X_test[cols_bin].values], axis=1)

**Regresión polinomial**

In [51]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

LinearRegression()

In [52]:
lr.intercept_

np.float64(19.369881697222002)

In [53]:
lr.coef_

array([-4.81015039,  0.22526256,  1.85084013, -0.25953366, -1.0362087 ,
       -2.40556049,  1.15494794, -0.56584566, -3.14384676])

**Interpretación de los coeficientes**

En este modelo la variable dependiente es qsec, que mide el tiempo que tarda el auto en recorrer un cuarto de milla. Por lo tanto, un coeficiente positivo indica que el tiempo aumenta (el auto es más lento) y uno negativo indica que el tiempo disminuye (el auto es más rápido). Según el modelo, más cilindros y mayor potencia aumentan el tiempo, mientras que mayor desplazamiento, más velocidades y más carburadores reducen el tiempo, es decir, se asocian con autos más rápidos. La transmisión manual también aumenta el tiempo respecto a la automática.

**Cálculo de estadístico R^2**

In [54]:
r2_train = lr.score(X_train_scaled, y_train)
r2_test = lr.score(X_test_scaled, y_test)

print("R2 train:", r2_train)
print("R2 test:", r2_test)

R2 train: 0.9848739209492603
R2 test: -1.381143823262783


**Regularización L2 (Ridge)**

In [55]:
lambdas = [0.01, 0.1, 1, 2, 10]
for l in lambdas:
    ridge = Ridge(alpha=l)
    ridge.fit(X_train_scaled, y_train)
    r2_train = ridge.score(X_train_scaled, y_train)
    r2_test = ridge.score(X_test_scaled, y_test)
    print("Lambda:", l)
    print("R2 train:", r2_train)
    print("R2 test:", r2_test)
    print("_____________________________")

Lambda: 0.01
R2 train: 0.984060975140091
R2 test: -0.849849238627314
_____________________________
Lambda: 0.1
R2 train: 0.9699933920412346
R2 test: 0.17612560774199193
_____________________________
Lambda: 1
R2 train: 0.9065906550163145
R2 test: 0.65054364821962
_____________________________
Lambda: 2
R2 train: 0.8678264464391621
R2 test: 0.6939012219836451
_____________________________
Lambda: 10
R2 train: 0.7054200851975375
R2 test: 0.6269528415638324
_____________________________


**Comparación de los R^2**

Al aplicar regularización L2, se observa que cuando λ es muy pequeño el modelo sobreajusta y tiene mal desempeño en prueba. Conforme aumenta λ, el R^2 de entrenamiento disminuye ligeramente, pero el R^2 de prueba mejora considerablemente. El mejor equilibrio se obtiene entre lambda 1 y 2, donde el modelo logra la mayor capacidad de generalización. Valores demasiado grandes de λ comienzan a reducir demasiado la complejidad del modelo, afectando también el desempeño.
_________________________________

**2.1 Regresión con variables categóricas (uso de variables dummy) con "mpg" como salida**

**Variables dummy**

In [83]:
df = df0.drop(columns=["model"]).copy()

In [84]:
df["cyl"].unique()

array([6, 4, 8])

In [85]:
df["cyl_6"] = (df["cyl"] == 6).astype(int)
df["cyl_8"] = (df["cyl"] == 8).astype(int)

df = df.drop(columns=["cyl"])

In [86]:
df["gear"].unique()

array([4, 3, 5])

In [87]:
df["gear_4"] = (df["gear"] == 4).astype(int)
df["gear_5"] = (df["gear"] == 5).astype(int)

df = df.drop(columns=["gear"])

In [88]:
df["carb"].unique()

array([4, 1, 2, 3, 6, 8])

In [89]:
df["carb_2"] = (df["carb"] == 2).astype(int)
df["carb_3"] = (df["carb"] == 3).astype(int)
df["carb_4"] = (df["carb"] == 4).astype(int)
df["carb_6"] = (df["carb"] == 6).astype(int)
df["carb_8"] = (df["carb"] == 8).astype(int)

df = df.drop(columns=["carb"])

**Train-test-split**

In [115]:
np.random.seed(42)
indices_train = np.random.choice(df.index, size=13, replace=False)

train = df.loc[indices_train]
test = df.drop(indices_train)

In [116]:
X_train = train.drop(columns=["mpg"])
y_train = train["mpg"]

X_test = test.drop(columns=["mpg"])
y_test = test["mpg"]

In [117]:
cols_num = ["disp","hp","drat","wt","qsec"]
cols_bin = [
    "vs","am",
    "cyl_6","cyl_8",
    "gear_4","gear_5",
    "carb_2","carb_3","carb_4","carb_6","carb_8"
]

**Regresión polinomial**

In [118]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

LinearRegression()

In [119]:
lr.intercept_

np.float64(8.437583352991734)

In [120]:
lr.coef_

array([-24.79998502, -11.76958061,   7.54952846,  23.22158712,
        -4.1353338 ,  -1.31404246,  12.69938525,  14.76987892,
        31.58133518, -12.71721499, -11.22299392,   7.22881924,
       -13.83709425, -17.49428277, -13.30457337,   2.08157945])

**Cálculo R^2**

In [121]:
print("R2 train:", lr.score(X_train_scaled, y_train))
print("R2 test:", lr.score(X_test_scaled, y_test))

R2 train: 1.0
R2 test: -2.6653856443994304


**Interpretación de coeficientes y R^2**

En este modelo con variables dummy, el intercepto representa el mpg esperado para un auto con 4 cilindros, 3 velocidades y 1 carburador (categorías base), manteniendo el resto de variables en su nivel promedio. Los coeficientes negativos de disp y hp indican que motores más grandes y potentes tienden a reducir el rendimiento en millas por galón, lo cual es coherente con la teoría. La transmisión manual (am) muestra un efecto positivo importante sobre el mpg, mientras que varias dummies de cilindros y carburadores presentan efectos grandes y algunos poco realistas, lo que sugiere inestabilidad en el modelo. Esto se confirma con los valores de R^2, pues el de entrenamiento es 1.0, lo cual es muy bueno, pero el R^2 de prueba es negativo (-2.66), lo que evidencia un sobreajuste severo y una mala capacidad de generalización.

**2.2 Regresión con variables categóricas (uso de variables dummy) con "qsec" como salida**|

**Train-test-split**

In [123]:
np.random.seed(42)
indices_train = np.random.choice(df.index, size=13, replace=False)

train = df.loc[indices_train]
test = df.drop(indices_train)

X_train = train.drop(columns=["qsec"])
y_train = train["qsec"]

X_test = test.drop(columns=["qsec"])
y_test = test["qsec"]

In [125]:
cols_num = ["disp","hp","drat","wt"]   

cols_bin = [
    "vs","am",
    "cyl_6","cyl_8",
    "gear_4","gear_5",
    "carb_2","carb_3","carb_4","carb_6","carb_8"
]

**Regresión polinomial**

In [128]:
lr = LinearRegression()
lr.fit(X_train[cols_num + cols_bin], y_train)

LinearRegression()

In [130]:
lr.intercept_

np.float64(31.530123611367177)

In [131]:
lr.coef_

array([-2.62204355e-02, -2.32894703e-03, -3.13867380e+00,  2.12125988e+00,
       -5.59737787e-01, -9.52629779e-01, -3.28306793e+00, -3.72932669e+00,
       -1.91479927e-01, -7.12133044e-01,  1.65581403e+00, -8.26561171e-01,
        4.91634779e-01, -1.38665472e+00,  6.74521681e-01])

In [132]:
print("R2 train:", lr.score(X_train[cols_num + cols_bin], y_train))
print("R2 test:", lr.score(X_test[cols_num + cols_bin], y_test))

R2 train: 1.0
R2 test: -0.5303320151068469


**Interpretación**

En este modelo el intercepto (31.53) representa el tiempo estimado en el cuarto de milla para el auto base (4 cilindros, 3 velocidades y 1 carburador), cuando las demás variables están en su nivel de referencia. Los coeficientes negativos en variables como hp, disp y algunas dummies indican que mayores niveles de potencia o ciertas configuraciones reducen el tiempo en el cuarto de milla (el auto es más rápido). En cambio, coeficientes positivos implican que el tiempo aumenta, es decir, el auto es más lento bajo esa configuración específica.

Respecto al R^2, el valor de entrenamiento es 1.0, lo que significa que el modelo explica el 100% de la variabilidad en los datos de entrenamiento. Sin embargo, el R^2 de prueba es negativo (-0.53), lo que indica que el modelo generaliza muy mal y presenta sobreajuste. En otras palabras, aprendió demasiado bien los datos de entrenamiento pero no funciona correctamente con datos nuevos.
_____________________________

**3.1 Comparación de R^2 de 1.1 y 2.1**

En el ejercicio 2.1 el R^2 de entrenamiento aumenta hasta 1.0, lo que significa que el modelo ajusta perfectamente los datos de entrenamiento. Sin embargo, el R^2 de prueba nos informa un sobreajuste severo, con un estadístico R^2 de -2.66. Esto sucede porque al agregar variables dummy aumentamos mucho la cantidad de parámetros en un dataset pequeño. El modelo se vuelve más flexible y aprende demasiado los datos de entrenamiento, pero pierde capacidad de generalización. Por otra parte, el modelo 1.1 tiene un R^2 de entrenamiento de 0.97, que resulta muy bueno pero con un R^2 de prueba negativo de -1.47. Sin embargo, con la penalización de Ridge estos valores pueden llegar hasta 0.90 y 0.75 respectivamente. En conclusión, aunque el modelo con dummies parece “mejor” en entrenamiento, en realidad es peor porque generaliza menos que el modelo del ejercicio 1.1.
__________

**3.2 Comparación de R^2 de 1.2 y 2.2**

El modelo del apartado 2.2 (con dummies) tiene un modelo que explica el 100% de la variabilidad, pero sin ninguna regularización tiene un sobreajuste fuerte de -0.53. Por otro laod, el modelo 1.2 también sobreajusta, pues tiene un R^2 de -1.38. Sin embargo, cuando aplicamos Ridge al modelo 1.2, el desempeño mejora drásticamente y logra un R^2 de entrenamiento de hasta 0.90 con un R^2 de prueba de 0.65. Esto muestra que la regularización es clave.
En conclusión, el mejor modelo no es el que tiene R² train más alto, sino el que logra mejor R^2 test después de regularizar.
__________